In [1]:
import pandas as pd
import numpy as np
import os
import logging

In [2]:
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit\\')

In [3]:
Bureau = pd.read_csv('data/External/bureau.csv')

In [4]:
BureauBal = pd.read_csv('data/External/bureau_balance.csv')

In [5]:
Bureau.dtypes

SK_ID_CURR                  int64
SK_ID_BUREAU                int64
CREDIT_ACTIVE              object
CREDIT_CURRENCY            object
DAYS_CREDIT                 int64
CREDIT_DAY_OVERDUE          int64
DAYS_CREDIT_ENDDATE       float64
DAYS_ENDDATE_FACT         float64
AMT_CREDIT_MAX_OVERDUE    float64
CNT_CREDIT_PROLONG          int64
AMT_CREDIT_SUM            float64
AMT_CREDIT_SUM_DEBT       float64
AMT_CREDIT_SUM_LIMIT      float64
AMT_CREDIT_SUM_OVERDUE    float64
CREDIT_TYPE                object
DAYS_CREDIT_UPDATE          int64
AMT_ANNUITY               float64
dtype: object

In [6]:
Bureau = Bureau.astype({'SK_ID_CURR': 'string', 'SK_ID_BUREAU': 'string'})

Here is the correct way to interpret the variables for a credit card:

- AMT_CREDIT_SUM: This is the total credit limit on the card.

- AMT_CREDIT_SUM_DEBT: This is the current balance, or how much you owe.

- AMT_CREDIT_SUM_LIMIT: This is the available credit, or how much you can still spend.

- For closed account (~ 1m) `AMT_CREDIT_SUM_LIMIT` and `AMT_CREDIT_SUM_DEBT` = 0 (mostly) 
- For active accounts (~ 600k) 
    - Consumer Credits (~ 300k) have zero  `AMT_CREDIT_SUM_LIMIT`
    - Credit Cards (~ 300k) 
        - 40k have `AMT_CREDIT_SUM` > 0 and `AMT_CREDIT_SUM_DEBT`= 0 and `AMT_CREDIT_SUM_LIMIT`=0.
        - 87k have > `AMT_CREDIT_SUM` = `AMT_CREDIT_SUM_DEBT` + `AMT_CREDIT_SUM_LIMIT`
        - 87k cases where `AMT_CREDIT_SUM` != `AMT_CREDIT_SUM_DEBT` + `AMT_CREDIT_SUM_LIMIT`
        - 1310 cases where `AMT_CREDIT_SUM` = 0 but `AMT_CREDIT_SUM_DEBT` > 0
        - 63k have > `AMT_CREDIT_SUM` and either of the two missing

In [7]:
# Set AMT_CREDIT_SUM_LIMIT to 0 for closed credits where it is null
Bureau.loc[(Bureau['CREDIT_ACTIVE'] == 'Closed') & (Bureau['AMT_CREDIT_SUM_LIMIT'].isnull()), 'AMT_CREDIT_SUM_LIMIT'] = 0

In [8]:
# Set AMT_CREDIT_SUM_DEBT to 0 for closed credits where it is null
Bureau.loc[(Bureau['CREDIT_ACTIVE'] == 'Closed') & (Bureau['AMT_CREDIT_SUM_DEBT'].isnull()), 'AMT_CREDIT_SUM_DEBT'] = 0

In [9]:
Bureau.loc[(Bureau['CREDIT_ACTIVE'] == 'Active') & (Bureau['AMT_CREDIT_SUM_LIMIT'].isnull()) & (Bureau['CREDIT_TYPE'] != 'Credit card'), 'AMT_CREDIT_SUM_LIMIT'] = 0

Calculate utilization rate for Credit cards to determine the average debit amount.

In [10]:
Bureau['UtilzRate'] = np.where(Bureau['AMT_CREDIT_SUM']>0, Bureau['AMT_CREDIT_SUM_DEBT'] / Bureau['AMT_CREDIT_SUM'], np.nan)

In [11]:
Bureau['Duration'] = Bureau['DAYS_CREDIT_ENDDATE'] - Bureau['DAYS_CREDIT'] 

In [12]:
Bureau['OverDraftFlag'] = 1 * (Bureau['AMT_CREDIT_SUM_DEBT'] > Bureau['AMT_CREDIT_SUM'])

In [13]:
Bureau['AnnuityPercent'] = np.where(Bureau['AMT_CREDIT_SUM_DEBT']>0, Bureau['AMT_ANNUITY'] / Bureau['AMT_CREDIT_SUM_DEBT'], np.nan)

In [14]:
def get_utilization_rate(x, fulldf):
    y = fulldf.loc[x.index]
    y = y[y['CREDIT_ACTIVE'] == 'Active']
    return np.mean(np.where(
                                y['AMT_CREDIT_SUM'] > 0,
                                y['AMT_CREDIT_SUM_DEBT'] / y['AMT_CREDIT_SUM'],
                                np.nan
                            ) )

 Bureau Balance

In [15]:
BureauBal.dtypes

SK_ID_BUREAU       int64
MONTHS_BALANCE     int64
STATUS            object
dtype: object

In [16]:
BureauBal = BureauBal.astype({'SK_ID_BUREAU': 'string'})

In [17]:
BureauBal.sort_values(by=['SK_ID_BUREAU', 'MONTHS_BALANCE'], inplace=True)

Impute data by going forward and backward.

In [18]:
BureauBal['STATUS_FILLED'] = BureauBal.groupby('SK_ID_BUREAU')['STATUS'].transform(
    lambda x: x.replace('X', np.nan).ffill(limit=3).bfill(limit=3)
)

C:\Users\domjo\AppData\Local\Temp\ipykernel_12784\2127701334.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.replace('X', np.nan).ffill(limit=3).bfill(limit=3)


In [19]:
BureauBal.groupby('SK_ID_BUREAU').last()#.reset_index()[['SK_ID_BUREAU', 'STATUS']].value_counts(dropna=False)

,MONTHS_BALANCE,STATUS,STATUS_FILLED
SK_ID_BUREAU,,,
5001709,0,C,C
5001710,0,C,C
5001711,0,X,0
5001712,0,C,C
5001713,0,X,None
...,...,...,...
6842884,0,C,C
6842885,0,5,5
6842886,0,C,C


In [20]:
BureauBal['STATUS_FILLED'].value_counts(dropna=False)

STATUS_FILLED
C      13735833
0       8705656
NaN     4488360
1        260330
5         67892
2         25590
3          9496
4          6768
Name: count, dtype: int64

In [21]:
BureauBal_Silver = BureauBal.groupby('SK_ID_BUREAU').agg(
    Duration_Balance=('MONTHS_BALANCE', lambda x: x.max() - x.min()),
    BureauBal_Last_Status=('STATUS_FILLED', 'last'),
    BureauBal_Days_With_NoDPD=('STATUS_FILLED', lambda x: sum(x == '0')),
 #  BureauBalDaysWithNoDPDRatio=('STATUS_FILLED', lambda x: sum(x == '0') / (x.max() - x.min())),
    BureauBal_Days_In_DPD1=('STATUS_FILLED', lambda x: sum(x == '1')),
    BureauBal_Days_In_DPD2=('STATUS_FILLED', lambda x: sum(x == '2')),
    BureauBal_Days_In_DPD3=('STATUS_FILLED', lambda x: sum(x == '3')),
    BureauBal_Days_In_DPD4=('STATUS_FILLED', lambda x: sum(x == '4')),
    BureauBal_Days_In_DPD5=('STATUS_FILLED', lambda x: sum(x == '5'))
).reset_index()

In [22]:
BureauBal_Silver['BureauBal_Days_In_DPD_ge1'] = BureauBal_Silver['BureauBal_Days_In_DPD1'] + BureauBal_Silver['BureauBal_Days_In_DPD2'] + BureauBal_Silver['BureauBal_Days_In_DPD3'] + BureauBal_Silver['BureauBal_Days_In_DPD4'] + BureauBal_Silver['BureauBal_Days_In_DPD5']

In [23]:
BureauBal_Silver['BureauBal_Days_In_DPD_ge4'] = BureauBal_Silver['BureauBal_Days_In_DPD4'] + BureauBal_Silver['BureauBal_Days_In_DPD5']

In [24]:
Bureau_Bronze = Bureau.merge(
    BureauBal_Silver,
    how='left',
    left_on='SK_ID_BUREAU',
    right_on='SK_ID_BUREAU'
)

In [26]:
Bureau_Silver = Bureau_Bronze.groupby('SK_ID_CURR').agg(

    BUREAU_TOTAL_CREDIT_COUNT=('SK_ID_BUREAU', 'count'),
    
    BUREAU_CREDIT_TYPE_TOTAL_COUNT=('CREDIT_TYPE', 'nunique'),
    BUREAU_CONSUMERCREDIT_COUNT= ('CREDIT_TYPE', lambda x: sum(x == 'Consumer credit')),
    BUREAU_CREDITCARD_COUNT= ('CREDIT_TYPE', lambda x: sum(x == 'Credit card')),
    BUREAU_CARLOAN_COUNT= ('CREDIT_TYPE', lambda x: sum(x == 'Car loan')),
    BUREAU_MORTGAGE_COUNT= ('CREDIT_TYPE', lambda x: sum(x == 'Mortgages')),

    BUREAU_TOTAL_ACTIVE_CREDIT_COUNT=('SK_ID_BUREAU', lambda x: sum(Bureau.loc[x.index, 'CREDIT_ACTIVE'] == 'Active')),
    BUREAU_PAST_BAD_CREDIT_COUNT=('SK_ID_BUREAU', lambda x: sum(Bureau.loc[x.index, 'CREDIT_ACTIVE'] == 'Bad debt')),
    BUREAU_PAST_SOLD_CREDIT_COUNT=('SK_ID_BUREAU', lambda x: sum(Bureau.loc[x.index, 'CREDIT_ACTIVE'] == 'Sold')),

    BUREAU_DAYS_TO_LAST_CREDIT=('DAYS_CREDIT', 'max'),
    BUREAU_AVG_DAYS_TO_CREDIT=('DAYS_CREDIT', lambda x: x.sort_values(ascending=True).diff().mean() ),
    BUREAU_AVG_DURATION=('Duration', 'mean'),
    BUREAU_MAX_DAYS_CREDIT_END=('DAYS_CREDIT_ENDDATE', 'max'),

    BUREAU_IS_PAST_OVERDUE_FLAG=('CREDIT_DAY_OVERDUE', lambda x: (x > 0).any() * 1),
    BUREAU_ACTIVE_UTILIZATION_RATIO=('UtilzRate', 'mean'),    

    BUREAU_MAX_CREDIT_OVERDUE_AMOUNT=('AMT_CREDIT_MAX_OVERDUE', 'max'),
    BUREAU_AVG_CREDIT_AMOUNT=('AMT_CREDIT_SUM', 'mean'),
    BUREAU_AVG_CREDIT_DEBT_AMOUNT=('AMT_CREDIT_SUM_DEBT', 'mean'),
    BUREAU_OVERDRAFT_FLAG_COUNT = ('OverDraftFlag', 'sum'),
    BUREAU_AVG_ANNUITY_PERCENT=('AnnuityPercent', 'mean'),

  # BUREAUBAL_LAST_STATUS_DPD5_FLAG=('BureauBalLastStatus', lambda x: (x == '5').any() * 1),
    BUREAUBAL_LAST_STATUS_DPD_GT_90_ID_FLAG=('BureauBal_Last_Status', lambda x: (x.isin(['4','5']).any() * 1)),
    BUREAUBAL_LAST_STATUS_DPD_GT_90_ID_COUNT=('BureauBal_Last_Status', lambda x: sum(x.isin(['4','5']) * 1)),

    BUREAUBAL_DAYS_IN_DPD_GT_90_ID_COUNT=('BureauBal_Days_In_DPD_ge4', lambda x: sum((x > 0)*1)),
    BUREAUBAL_DAYS_IN_DPD_GT_90_DAYS_COUNT=('BureauBal_Days_In_DPD_ge4', 'sum'),

    BUREAUBAL_DAYS_IN_DPD_GT_0_ID_COUNT=('BureauBal_Days_In_DPD_ge1', lambda x: sum((x > 0)*1)),
    BUREAUBAL_DAYS_IN_DPD_GT_0_DAYS_COUNT=('BureauBal_Days_In_DPD_ge1', 'sum'),
)

In [29]:
Bureau_Silver.to_parquet("a:\\Dominic\\My Projects\\Home Credit\\src\\intermediate\\bureau_processed.parquet")